# 05 - Modèle FEM d'élasticité

Ce notebook résout un problème d'élasticité linéaire sur un tétraèdre. Trois nœuds sont bloqués et une force est appliquée au dernier nœud.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from brain_mri_fem.fem import solve_tetrahedral_elasticity

nodes = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=float)
elements = np.array([[0, 1, 2, 3]], dtype=int)
forces = np.zeros((4, 3), dtype=float)
forces[3] = (0.0, 0.0, -1.0)
result = solve_tetrahedral_elasticity(nodes, elements, fixed_nodes=np.array([0, 1, 2]), forces=forces)
displacement = result["displacement"]
print("Déplacements:\n", displacement)
print("Norme maximale:", float(np.linalg.norm(displacement, axis=1).max()))

In [ ]:
scale = 0.25 / max(float(np.linalg.norm(displacement, axis=1).max()), 1e-12)
deformed = nodes + scale * displacement
fig = plt.figure(figsize=(7, 6))
axis = fig.add_subplot(111, projection="3d")
for point in nodes:
    axis.scatter(*point, color="steelblue", s=50)
for point in deformed:
    axis.scatter(*point, color="darkorange", s=50)
for start, end in ((0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)):
    axis.plot(*zip(nodes[start], nodes[end]), color="steelblue", alpha=0.5)
    axis.plot(*zip(deformed[start], deformed[end]), color="darkorange")
axis.set_title("Tétraèdre initial (bleu) et déformé (orange)")
fig.tight_layout()
output = REPO_ROOT / "results/figures/fem"
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "tetrahedral_deformation.png", dpi=150)
(REPO_ROOT / "results/demo").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "results/demo/fem_notebook.json").write_text(json.dumps({"displacement": displacement.tolist(), "scale": scale}, indent=2), encoding="utf-8")
plt.show()